In [1]:
# Import numerical libraries
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

# Import helpers
import itertools
from tqdm import tqdm
from collections import defaultdict
from pathlib import Path
import json
import os
import yaml


# Import experiment handler
from experiment_logging import Experiment

# Import machine learning
from dataset_creator import create_dataset, dynamics_sincos, time_series_generator
from time_series.data_handlers import TimeSeriesData
from time_series.models import *
import optuna

2026-03-16 14:43:11.268 | INFO     | time_series.config:<module>:13 - PROJ_ROOT path is: /home/james/Repo/PhD Repo/time_series_clustering
/home/james/Repo/PhD Repo/time_series_clustering/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
with open("experiment_configs.yaml") as config_file:
    config = yaml.safe_load(config_file)
    config_file.close()

In [4]:
models = {}

for m in config["models"]:
    if config["models"][m]["model_name"] == "KernelRidgeRegression":
        models[m] = dict(
            model=KernelRidgeRegression,
            **config["models"][m]
        )
    elif config["models"][m]["model_name"] == "RascuttiModel":
        models[m] = dict(
            model=RascuttiModel,
            **config["models"][m]
        )
    elif config["models"][m]["model_name"] == "EigenRascuttiModel":
        models[m] = dict(
            model=EigenRascuttiModel,
            **config["models"][m]
        )
    else:
        raise ValueError(f"{config["models"][m]["model_name"]} does not exist")

In [5]:
def is_sweep(x):
    # Exclusions
    if type(x) in [dict, str]:
        return False
    return hasattr(x, "__iter__")

default_values = config["default_values"]

experiments = []

for experiment_name, experiment_config in config["experiments"].items():
    experiment_configs_processed = dict()
    experiment_configs_processed.update(default_values)
    experiment_configs_processed.update(
        dict(
            name = experiment_config["name"],
            model = models[experiment_config["model"]]
        )
    )
    if "sweeps" in experiment_config:
        sweeps = experiment_config["sweeps"]
        for param, param_conf in sweeps.items():
            if param_conf["type"] == "int":
                experiment_configs_processed[param] = np.arange(param_conf["min"], param_conf["max"], param_conf["step"])
            elif param_conf["type"] == "float":
                experiment_configs_processed[param] = np.linspace(param_conf["min"], param_conf["max"], param_conf["n_steps"])
                
    names = experiment_configs_processed.keys()
    vals = experiment_configs_processed.values()

    for res in itertools.product(*list(map(lambda x: x if is_sweep(x) else [x], vals))):
        experiments.append(dict(zip(names, res)))

In [ ]:
experiment_trackers = {}

counter = 0
for experiment in tqdm(experiments):
    # Initialise experiment
    exp_name = experiment["name"] + " - " + experiment["model"]["model_name"]
    if exp_name not in experiment_trackers:
        experiment_trackers[exp_name] = Experiment(
            exp_name,
            "experiments"
        )

    experiment_tracker = experiment_trackers[exp_name]

    # Extract config
    conf = dict(experiment)
    conf.pop("model")
    conf.update({"model":experiment["model"]["model_name"]})

    experiment_tracker.add_config(**conf)

    # Set up theta sweeps
    theta_vals = np.linspace(
        experiment["theta_min"],
        experiment["theta_max"],
        experiment["n_thetas"]
    )

    # Run repeats
    similarities_all = []
    for r in range(experiment["n_repeat"]):
        ref_data = create_dataset(
            experiment["theta_ref"],
            n_points=experiment["n_points"],
            n_correlated_dimensions=experiment["n_correlated_dims"],
            n_uncorrelated_dimensions=experiment["n_uncorrelated_dims"],
            noise=experiment["noise"],
        )

        ref_dataset = TimeSeriesData(
            X=ref_data[:-1],
            y=ref_data[1:],
            lag=1,
            train_val_test_split=[0.5, 0.3, 0.2],
        )

        # theta datasets
        datasets = []
        for theta in theta_vals:
            data = create_dataset(
                theta,
                n_points=experiment["n_points"],
                n_correlated_dimensions=experiment["n_correlated_dims"],
                n_uncorrelated_dimensions=experiment["n_uncorrelated_dims"],
                noise=experiment["noise"],
            )

            datasets.append(
                TimeSeriesData(
                    X=data[:-1],
                    y=data[1:],
                    lag=1,
                    train_val_test_split=[0.5, 0.3, 0.2],
                )
            )

        # similarities
        sims = np.zeros(len(theta_vals))
        for i, ds in enumerate(datasets):
            # hyperparams
            model_class = experiment["model"]["model"]
            model_params = experiment["model"]["parameters"]
            model_hparams_owned = experiment["model"]["hyperparameters"]["owned"]
            model_hparams_shared = experiment["model"]["hyperparameters"]["shared"]
            

            def objective(trial):
                model1_params = dict(model_params)
                model2_params = dict(model_params)

                for param in model_hparams_owned:
                    if model_hparams_owned[param]["type"] == "float":
                        model1_params[param] = trial.suggest_float(
                            param + "_model1", 
                            model_hparams_owned[param]["min"], 
                            model_hparams_owned[param]["max"]
                        )

                        model2_params[param] = trial.suggest_float(
                            param + "_model2", 
                            model_hparams_owned[param]["min"], 
                            model_hparams_owned[param]["max"]
                        )
                    
                    else:
                        raise TypeError("Hparam type not supported yet")

                for param in model_hparams_shared:
                    if model_hparams_shared[param]["type"] == "float":
                        param_opt = trial.suggest_float(
                            param + "_shared", 
                            model_hparams_shared[param]["min"], 
                            model_hparams_shared[param]["max"]
                        )

                        model1_params[param] = param_opt
                        model2_params[param] = param_opt
                    
                    else:
                        raise TypeError("Hparam type not supported yet")


                mse = 0.0

                X_tr, y_tr = ref_dataset.train_data()
                X_va, y_va = ref_dataset.val_data()

                model1 = model_class(**model1_params)
                model1.fit(X_tr, y_tr)
                mse += np.mean((model1.predict(X_va) - y_va) ** 2)

                X_tr, y_tr = ds.train_data()
                X_va, y_va = ds.val_data()

                model2 = model_class(**model2_params)
                model2.fit(X_tr, y_tr)
                mse += np.mean((model2.predict(X_va) - y_va) ** 2)

                return mse

            study = optuna.create_study()
            study.optimize(objective, n_trials=experiment["n_trials"], n_jobs=-1)
            best_params = study.best_params

            model1_best_params = dict(model_params)
            model2_best_params = dict(model_params)

            for param in best_params:
                if "_model1" in param:
                    model1_best_params[param.removesuffix("_model1")] = best_params[param]
                elif "_model2" in param:
                    model2_best_params[param.removesuffix("_model2")] = best_params[param]
                elif "_shared" in param:
                    model1_best_params[param.removesuffix("_shared")] = best_params[param]
                    model2_best_params[param.removesuffix("_shared")] = best_params[param]
                else:
                    pass

            X_ref, y_ref = ref_dataset.full_data()
            model_ref = model_class(**model1_best_params)
            model_ref.fit(X_ref, y_ref)
            ip11 = model_ref.inner_product(model_ref)

            X, y = ds.full_data()
            model = model_class(**model2_best_params)
            model.fit(X, y)

            ip12 = model_ref.inner_product(model)
            ip22 = model.inner_product(model)

            sims[i] = ip12/np.sqrt(ip11 * ip22)
        
        similarities_all.append(sims)

    sims = np.stack(similarities_all)

    experiment_tracker.add_result(
        **{f"result_{counter}": dict(
            noise=experiment["noise"],
            theta_ref=experiment["theta_ref"],
            theta_values=theta_vals.tolist(),
            mean_similarities=sims.mean(axis=0).tolist(),
            std_similarities=sims.std(axis=0).tolist(),
            n_repeat=experiment["n_repeat"],
            best_params = best_params
        )}
    ) 
    counter += 1

  0%|          | 0/2 [00:00<?, ?it/s]/home/james/Repo/PhD Repo/time_series_clustering/venv/lib/python3.12/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
